In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import geopandas as gpd
import glob
import os
import matplotlib.pyplot as plt

In [ ]:
# ---------------- CONFIG ----------------
folder = "/content/drive/MyDrive/Congestion_LISA"
files = glob.glob(os.path.join(folder, "*.geojson")) + glob.glob(os.path.join(folder, "*.json"))
print(f"Found {len(files)} files")

dfs = []

# ---------------- LOAD SNAPSHOTS ----------------
for f in files:
    try:
        gdf = gpd.read_file(f)

        # Parse timestamp from filename
        # Example: congestion_lisa_2026-03-17_07_00_00
        base = os.path.splitext(os.path.basename(f))[0]
        stamp = base.replace("congestion_lisa_", "")

        try:
            dt = pd.to_datetime(stamp, format="%Y-%m-%d_%H_%M_%S")
        except:
            dt = pd.NaT

        gdf["snapshot_file"] = base
        gdf["Date"] = dt

        dfs.append(gdf)

    except Exception as e:
        print(f"Skipping {f}: {e}")

combined = pd.concat(dfs, ignore_index=True)
print(f"Combined rows: {len(combined)}")
print("Columns found:")
print(combined.columns.tolist())

# ---------------- IDENTIFY REGION FIELD ----------------
region_candidates = ["region", "Region", "city", "City"]
region_field = next((c for c in region_candidates if c in combined.columns), None)

if region_field is None:
    raise ValueError(f"No city/region field found. Columns available: {combined.columns.tolist()}")

combined["Region"] = combined[region_field].astype(str).str.replace("_", " ", regex=False)

# ---------------- CLEAN LABEL FIELD ----------------
if "congestionLabel" not in combined.columns:
    raise ValueError(f"'congestionLabel' not found. Columns available: {combined.columns.tolist()}")

combined["congestionLabel"] = combined["congestionLabel"].astype(str).str.strip()

valid_labels = ["Free Flowing", "Moderate Flow", "Congested"]
combined = combined[combined["congestionLabel"].isin(valid_labels)].copy()

# ---------------- CLASSIFY AM / PM ----------------
combined["Hour"] = combined["Date"].dt.hour

def classify_period(hour):
    if pd.isna(hour):
        return None
    if hour in [7, 8, 9]:
        return "AM"
    elif hour in [16, 17, 18]:
        return "PM"
    else:
        return None

combined["Travel Period"] = combined["Hour"].apply(classify_period)

# ============================================================
# LISA CLUSTER MEMBERSHIP BY ROAD CLASS
# ============================================================

# ---------------- CHECK REQUIRED FIELDS ----------------
# Expected fields from LISA outputs:
# highway = OSM road class
# lisa_cluster = Local Moran cluster label
required_lisa_fields = ["highway", "lisa_cluster"]

missing_lisa_fields = [c for c in required_lisa_fields if c not in combined.columns]
if missing_lisa_fields:
    raise ValueError(
        f"Missing required fields for LISA road-class summary: {missing_lisa_fields}. "
        f"Available columns: {combined.columns.tolist()}"
    )

# ---------------- CLEAN ROAD CLASS FIELD ----------------
# Some OSM highway values may be stored as lists/strings like "['primary', 'secondary']"
# This converts them into cleaner, consistent text categories.

def clean_highway_class(value):
    if pd.isna(value):
        return "Unknown"

    value = str(value).strip()

    # Remove common list/string formatting artifacts
    value = (
        value.replace("[", "")
             .replace("]", "")
             .replace("'", "")
             .replace('"', "")
    )

    # If multiple road classes exist, keep the first one as the primary class
    if "," in value:
        value = value.split(",")[0].strip()

    if value == "" or value.lower() == "nan":
        return "Unknown"

    return value

combined["Road Class"] = combined["highway"].apply(clean_highway_class)

# ---------------- CLEAN LISA CLUSTER FIELD ----------------
combined["LISA Cluster"] = combined["lisa_cluster"].astype(str).str.strip()

valid_lisa_clusters = [
    "High-High",
    "Low-Low",
    "High-Low",
    "Low-High",
    "Not Significant"
]

combined_lisa = combined[combined["LISA Cluster"].isin(valid_lisa_clusters)].copy()

print("LISA cluster values retained:")
print(combined_lisa["LISA Cluster"].value_counts())

print("\nRoad classes found:")
print(combined_lisa["Road Class"].value_counts())

# ============================================================
# MAJORITY SIGNIFICANT LISA CLUSTER BY CITY AND ROAD CLASS
# Excludes "Not Significant" to examine only meaningful local
# spatial association patterns.
# ============================================================

# Keep only statistically significant LISA classes
significant_clusters = ["High-High", "Low-Low", "High-Low", "Low-High"]

sig_lisa = combined_lisa[
    combined_lisa["LISA Cluster"].isin(significant_clusters)
].copy()

print(f"Significant LISA segment observations retained: {len(sig_lisa):,}")
print(sig_lisa["LISA Cluster"].value_counts())

# Count significant cluster observations by city, road class, and LISA type
sig_counts = (
    sig_lisa
    .groupby(["Region", "Road Class", "LISA Cluster"])
    .size()
    .reset_index(name="Count")
)

# Total significant observations by city + road class
sig_counts["Total Significant"] = sig_counts.groupby(
    ["Region", "Road Class"]
)["Count"].transform("sum")

# Percent of significant LISA observations represented by each cluster type
sig_counts["Percent of Significant"] = (
    sig_counts["Count"] / sig_counts["Total Significant"] * 100
).round(1)

# Identify majority significant LISA cluster for each city + road class
idx = sig_counts.groupby(["Region", "Road Class"])["Count"].idxmax()
majority_sig = sig_counts.loc[idx].copy()

majority_sig = majority_sig.rename(columns={
    "Region": "City",
    "LISA Cluster": "Majority Significant LISA Cluster"
})

majority_sig = majority_sig.sort_values(
    ["City", "Road Class"]
).reset_index(drop=True)

# Save majority table
majority_out = "/content/drive/MyDrive/majority_significant_lisa_cluster_by_city_road_class.csv"
majority_sig.to_csv(majority_out, index=False)

print("\nMajority significant LISA cluster by city and road class:")
print(majority_sig)

print(f"\nSaved:\n{majority_out}")

# ============================================================
# PLOT: MAJORITY SIGNIFICANT LISA CLUSTER BY ROAD CLASS AND CITY
# ============================================================

import matplotlib.patches as mpatches
import numpy as np

# Optional: limit to major/interpretable road classes
major_road_classes = [
    "motorway",
    "motorway_link",
    "trunk",
    "trunk_link",
    "primary",
    "primary_link",
    "secondary",
    "secondary_link",
    "tertiary",
    "tertiary_link",
    "residential",
    "unclassified"
]

plot_df = majority_sig[
    majority_sig["Road Class"].isin(major_road_classes)
].copy()

# Keep only road classes that appear
road_order = [rc for rc in major_road_classes if rc in plot_df["Road Class"].unique()]
city_order = sorted(plot_df["City"].unique())

# Map clusters to numeric/color values
cluster_order = ["High-High", "Low-Low", "High-Low", "Low-High"]
cluster_to_num = {cluster: i for i, cluster in enumerate(cluster_order)}

plot_df["cluster_num"] = plot_df["Majority Significant LISA Cluster"].map(cluster_to_num)

# Pivot to city x road class matrix
heatmap_data = plot_df.pivot(
    index="City",
    columns="Road Class",
    values="cluster_num"
).reindex(index=city_order, columns=road_order)

# Colors consistent with LISA categories
cluster_colors = {
    "High-High": "#C0392B",
    "Low-Low": "#2E86C1",
    "High-Low": "#E67E22",
    "Low-High": "#27AE60"
}

from matplotlib.colors import ListedColormap, BoundaryNorm

cmap = ListedColormap([cluster_colors[c] for c in cluster_order])
norm = BoundaryNorm(np.arange(-0.5, len(cluster_order) + 0.5, 1), cmap.N)

fig, ax = plt.subplots(figsize=(14, 5))

im = ax.imshow(heatmap_data, cmap=cmap, norm=norm, aspect="auto")

ax.set_title("Majority Significant LISA Cluster by City and Road Class", fontsize=15)
ax.set_xlabel("Road Class", fontsize=12)
ax.set_ylabel("City", fontsize=12)

ax.set_xticks(np.arange(len(road_order)))
ax.set_xticklabels(road_order, rotation=45, ha="right")

ax.set_yticks(np.arange(len(city_order)))
ax.set_yticklabels(city_order)

# Add cell labels
for i, city in enumerate(city_order):
    for j, road_class in enumerate(road_order):
        value = heatmap_data.loc[city, road_class]
        if pd.notna(value):
            cluster_label = cluster_order[int(value)]
            short_label = {
                "High-High": "HH",
                "Low-Low": "LL",
                "High-Low": "HL",
                "Low-High": "LH"
            }[cluster_label]
            ax.text(j, i, short_label, ha="center", va="center", color="white", fontsize=9, fontweight="bold")

# Legend
legend_patches = [
    mpatches.Patch(color=cluster_colors[c], label=c)
    for c in cluster_order
]

ax.legend(
    handles=legend_patches,
    title="Majority Significant\nLISA Cluster",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)

plt.tight_layout()

majority_chart_out = "/content/drive/MyDrive/majority_significant_lisa_cluster_by_city_road_class.png"
plt.savefig(majority_chart_out, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved chart:\n{majority_chart_out}")

# ============================================================
# DIRECT HYPOTHESIS CHECK:
# Percent of significant LISA observations that are High-High
# by city and road class
# ============================================================

hh_share = (
    sig_counts
    .pivot_table(
        index=["Region", "Road Class"],
        columns="LISA Cluster",
        values="Count",
        fill_value=0
    )
    .reset_index()
)

# Ensure all significant cluster columns exist
for c in significant_clusters:
    if c not in hh_share.columns:
        hh_share[c] = 0

hh_share["Total Significant"] = hh_share[significant_clusters].sum(axis=1)

hh_share["High-High Share of Significant (%)"] = (
    hh_share["High-High"] / hh_share["Total Significant"] * 100
).round(1)

hh_share = hh_share.rename(columns={"Region": "City"})

hh_share = hh_share.sort_values(
    ["City", "Road Class"]
).reset_index(drop=True)

hh_share_out = "/content/drive/MyDrive/high_high_share_of_significant_by_city_road_class.csv"
hh_share.to_csv(hh_share_out, index=False)

print("\nHigh-High share of significant LISA observations by city and road class:")
print(hh_share[[
    "City",
    "Road Class",
    "High-High",
    "Total Significant",
    "High-High Share of Significant (%)"
]])

print(f"\nSaved:\n{hh_share_out}")

# ============================================================
# CHART: HIGH-HIGH SHARE OF SIGNIFICANT LISA OBSERVATIONS
# BY ROAD CLASS, AGGREGATED ACROSS CITIES
# ============================================================

hh_road = (
    sig_lisa
    .groupby(["Road Class", "LISA Cluster"])
    .size()
    .reset_index(name="Count")
)

hh_road_pivot = hh_road.pivot_table(
    index="Road Class",
    columns="LISA Cluster",
    values="Count",
    fill_value=0
)

for c in significant_clusters:
    if c not in hh_road_pivot.columns:
        hh_road_pivot[c] = 0

hh_road_pivot["Total Significant"] = hh_road_pivot[significant_clusters].sum(axis=1)
hh_road_pivot["High-High Share of Significant (%)"] = (
    hh_road_pivot["High-High"] / hh_road_pivot["Total Significant"] * 100
).round(1)

hh_road_pivot = hh_road_pivot.loc[
    [rc for rc in major_road_classes if rc in hh_road_pivot.index]
]

ax = hh_road_pivot["High-High Share of Significant (%)"].plot(
    kind="bar",
    figsize=(12, 6)
)

plt.title("High-High Share of Significant LISA Clusters by Road Class", fontsize=15)
plt.xlabel("Road Class", fontsize=12)
plt.ylabel("High-High Share of Significant LISA Observations (%)", fontsize=12)
plt.xticks(rotation=45, ha="right")
plt.ylim(0, 100)
plt.tight_layout()

hh_road_chart_out = "/content/drive/MyDrive/high_high_share_by_road_class.png"
plt.savefig(hh_road_chart_out, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved chart:\n{hh_road_chart_out}")